# Chapter 04. Naive Bayes로 도서 카테고리 분류하기

## 실습 1. 필요한 라이브러리 확인하기

### 📌 학습 목표 및 개념
* **개념:** Chapter 04 머신러닝 분류 모델링에 필요한 주요 패키지(`pandas`, `scikit-learn`, `matplotlib`, `joblib`)를 불러옵니다.
* **학습 목적:** 데이터 분할, 모델 학습, 평가 지표 계산을 위한 환경을 준비합니다.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ 기본 라이브러리 불러오기 완료")

✅ 기본 라이브러리 불러오기 완료


### 💡 실습 결과 상세 정리
* 머신러닝 모델링에 필요한 핵심 모듈들을 정상적으로 임포트했습니다.

## 실습 2. Chapter 01 전처리 데이터 불러오기

### 📌 학습 목표 및 개념
* **개념:** Chapter 01에서 정제하여 저장한 `book_bestseller_clean.csv` 데이터를 불러와 기본 정보 및 결측치를 점검합니다.
* **학습 목적:** 분류 모델의 입력 데이터로 사용할 상품명과 분야 데이터를 확인합니다.

In [2]:
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())
print("분야 결측치:", df_books["분야"].isna().sum())

df_books[["상품명", "분야"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0
분야 결측치: 0


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설
5,초역 부처의 말,인문
6,청춘의 독서(특별증보판),인문
7,어른의 행복은 조용하다,시/에세이
8,채식주의자,소설
9,단 한 번의 삶(강물에디션 활판인쇄 한정판),시/에세이


### 💡 실습 결과 상세 정리
* 전처리된 베스트셀러 데이터셋을 문제없이 불러왔으며, 상품명과 분야 컬럼의 정상 유무를 확인했습니다.

## 실습 3. 분류에 사용할 데이터 정리하기

### 📌 학습 목표 및 개념
* **개념:** 전체 데이터에서 분류 모델의 입력값($X$)으로 사용할 `상품명`과 정답($y$)으로 사용할 `분야` 컬럼만 별도로 추출하여 정제합니다.
* **학습 목적:** 원본 데이터를 보호하고, 공백 제거 및 결측치 처리를 진행하여 깔끔한 모델링용 데이터셋을 준비합니다.

In [3]:
# 1. 모델링에 필요한 컬럼만 복사
df_model = df_books[["상품명", "분야"]].copy()

# 2. 문자열 정제 및 공백 제거
df_model["상품명"] = df_model["상품명"].fillna("").astype(str).str.strip()
df_model["분야"] = df_model["분야"].fillna("").astype(str).str.strip()

# 3. 빈 값 제외 및 인덱스 재정렬
df_model = df_model[
    (df_model["상품명"] != "") & 
    (df_model["분야"] != "")
].reset_index(drop=True)

print("모델링 데이터 크기:", df_model.shape)
df_model.head()

모델링 데이터 크기: (199, 2)


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설


### 💡 실습 결과 상세 정리
* 원본 DataFrame의 손상을 방지하기 위해 복사본(`df_model`)을 만들어 정제 작업을 진행했습니다[cite: 1].
* `상품명`과 `분야` 컬럼의 양끝 공백 및 빈 값을 깔끔하게 정리하여 모델 학습 준비를 마쳤습니다[cite: 1].

## 실습 4. 분야 값 확인하기

### 📌 학습 목표 및 개념
* **개념:** 분류 모델의 정답(Target Label)으로 사용할 `분야` 컬럼의 고유값 개수와 각 분야별 데이터 분포를 확인합니다.
* **학습 목적:** 클래스 불균형(Class Imbalance) 여부를 파악하여 향후 학습 및 평가 시 주의할 항목을 점검합니다.

In [4]:
# 1. 분야 종류 수 확인
print("분야 종류 수:", df_model["분야"].nunique())

# 2. 분야별 데이터 개수 확인 (상위 20개)
print("\n--- 분야별 개수 (상위 20개) ---")
print(df_model["분야"].value_counts().head(20))

# 3. 분야별 비율 확인 (%)
print("\n--- 분야별 비율 (%) ---")
print((df_model["분야"].value_counts(normalize=True) * 100).round(2).head(20))

분야 종류 수: 16

--- 분야별 개수 (상위 20개) ---
분야
소설         48
인문         27
경제/경영      26
시/에세이      25
자기계발       17
외국어        15
어린이(초등)    10
정치/사회       7
청소년         5
과학          5
역사/문화       5
가정/육아       3
요리          2
만화          2
예술/대중문화     1
컴퓨터/IT      1
Name: count, dtype: int64

--- 분야별 비율 (%) ---
분야
소설         24.12
인문         13.57
경제/경영      13.07
시/에세이      12.56
자기계발        8.54
외국어         7.54
어린이(초등)     5.03
정치/사회       3.52
청소년         2.51
과학          2.51
역사/문화       2.51
가정/육아       1.51
요리          1.01
만화          1.01
예술/대중문화     0.50
컴퓨터/IT      0.50
Name: proportion, dtype: float64


### 💡 실습 결과 상세 정리
* 전체 데이터 내 분야의 종류 수와 각 분야별 고유 데이터 빈도 및 비율을 확인했습니다[cite: 1].
* 특정 분야에 데이터가 치우쳐 있는지(클래스 불균형) 파악하여 모델 평가 기준 설정에 참고합니다[cite: 1].

## 실습 5. 클래스 불균형 이해하기

### 📌 학습 목표 및 개념
* **개념:** 전체 데이터셋에서 특정 분야의 데이터 수가 다른 분야에 비해 월등히 많거나 적은 현상을 클래스 불균형(Class Imbalance)이라고 합니다.
* **학습 목적:** 데이터의 불균형 분포가 모델의 학습 및 평가에 미치는 영향을 이해하고, 데이터를 함부로 삭제하지 않으면서 올바르게 분석하는 기준을 정립합니다.

---

### 💡 클래스 불균형 다루기 원칙
* **무조건적인 삭제 지양:** 데이터가 적다고 해서 무조건 억지로 줄이거나 삭제하지 않습니다.
* **다각도 평가 지표 활용:** 전체 Accuracy(정확도)만 보고 판단하지 않고, 분야별 Precision, Recall, F1-score 및 오분류 사례를 함께 살펴봅니다.

In [5]:
# 클래스 불균형 확인 및 분석 원칙 안내 메시지
print("⚠️ 클래스 불균형(Class Imbalance) 시 주의사항:")
print("1. 특정 분야 데이터가 적다고 해서 무조건 삭제하거나 임의로 맞추지 않습니다.")
print("2. 단순 Accuracy 외에 분야별 Precision, Recall, F1-score를 함께 검토해야 합니다.")

⚠️ 클래스 불균형(Class Imbalance) 시 주의사항:
1. 특정 분야 데이터가 적다고 해서 무조건 삭제하거나 임의로 맞추지 않습니다.
2. 단순 Accuracy 외에 분야별 Precision, Recall, F1-score를 함께 검토해야 합니다.


### 💡 실습 결과 상세 정리
* 클래스 불균형의 개념을 파악했으며, 이후 모델 평가 시 종합적인 평가 지표를 활용해 모델을 진단하기로 정리했습니다.

## 실습 6. 입력 데이터 X와 정답 y 정의하기

### 📌 학습 목표 및 개념
* **개념:** 머신러닝 지도학습(Supervised Learning)에 사용할 독립변수(Feature) $X$와 종속변수(Target) $y$를 정의합니다.
* **학습 목적:** 도서 제목(`상품명`)을 입력값 $X$로, 도서 카테고리(`분야`)를 정답 $y$로 분리하여 모델에 전달할 입출력 구조를 갖춥니다.

In [6]:
# 입력 X (상품명)와 정답 y (분야) 정의
X = df_model["상품명"]
y = df_model["분야"]

print("X 형태 (도서 제목 수):", X.shape)
print("y 형태 (정답 분야 수):", y.shape)

print("\n--- X 샘플 (상위 5개) ---")
print(X.head())

print("\n--- y 샘플 (상위 5개) ---")
print(y.head())

X 형태 (도서 제목 수): (199,)
y 형태 (정답 분야 수): (199,)

--- X 샘플 (상위 5개) ---
0        소년이 온다
1            모순
2    결국 국민이 합니다
3           혼모노
4            급류
Name: 상품명, dtype: str

--- y 샘플 (상위 5개) ---
0       소설
1       소설
2    정치/사회
3       소설
4       소설
Name: 분야, dtype: str


### 💡 실습 결과 상세 정리
* 지도학습 분류 모델 생성을 위해 입력 데이터 $X$와 라벨 데이터 $y$를 차원 및 개수를 맞춰 정확히 분리했습니다.

## 실습 7. 입력 X와 정답 y 이해하기

### 📌 학습 목표 및 개념
* **개념:** 지도학습(Supervised Learning)에서 모델의 입력 데이터($X$)와 맞혀야 하는 정답 라벨($y$)의 관계를 정리합니다.
* **학습 목적:** `상품명`을 독립변수 $X$로, `분야`를 종속변수 $y$로 지정하여 머신러닝 데이터 구조로 분리합니다.

In [7]:
# X와 y 정의
X = df_model["상품명"]
y = df_model["분야"]

print("X 샘플 (상위 5개):")
print(X.head())

print("\ny 샘플 (상위 5개):")
print(y.head())

X 샘플 (상위 5개):
0        소년이 온다
1            모순
2    결국 국민이 합니다
3           혼모노
4            급류
Name: 상품명, dtype: str

y 샘플 (상위 5개):
0       소설
1       소설
2    정치/사회
3       소설
4       소설
Name: 분야, dtype: str


### 💡 실습 결과 상세 정리
* `상품명`을 모델 입력값 $X$로, `분야`를 정답 라벨 $y$로 분리하여 지도학습 데이터를 완성했습니다[cite: 1].

## 실습 8. 지도학습과 분류 이해하기

### 📌 학습 목표 및 개념
* **개념:** 지도학습(Supervised Learning) 중 정답 데이터가 범주형(Categorical)인 '분류(Classification)' 문제의 구조를 이해합니다.
* **학습 목적:** 도서 제목($X$)을 보고 미리 정의된 분야 카테고리($y$) 중 하나를 맞히는 머신러닝 프로세스의 원리를 정립합니다.

---

### 💡 지도학습 및 분류 구조
* **지도학습:** 입력 데이터($X$)와 정답 라벨($y$)을 함께 전달하여 학습시키는 방법입니다.
* **분류:** 연속된 숫자를 예측하는 회귀(Regression)와 달리, 여러 지정된 카테고리(클래스) 중 하나를 예측하는 문제입니다.

In [8]:
# 지도학습 및 분류 문제 정의 확인 메시지
print("💡 지도학습 분류 문제 정의:")
print("1. 입력(X): 도서 상품명 (텍스트)")
print("2. 정답(y): 도서 분야 (카테고리 라벨)")
print("3. 목표: 새로운 도서 제목이 들어왔을 때 가장 가능성이 높은 분야 예측")

💡 지도학습 분류 문제 정의:
1. 입력(X): 도서 상품명 (텍스트)
2. 정답(y): 도서 분야 (카테고리 라벨)
3. 목표: 새로운 도서 제목이 들어왔을 때 가장 가능성이 높은 분야 예측


### 💡 실습 결과 상세 정리
* 머신러닝 관점에서 이번 실습이 도서 제목($X$)으로 분야($y$)를 예측하는 **다중 클래스 분류(Multi-class Classification)** 문제임을 명확히 정의했습니다.

## 실습 9. 왜 train과 test를 나눌까요?

### 📌 학습 목표 및 개념
* **개념:** 전체 데이터를 모델 학습용(Train Set)과 최종 평가용(Test Set)으로 분리해야 하는 이유를 이해합니다.
* **학습 목적:** 이미 학습한 데이터로 모델을 평가할 때 발생하는 일반화 성능 착시를 방지하고, 처음 보는 실제 데이터에서의 성능을 객관적으로 측정합니다.

---

### 💡 Train / Test 분리 이유
* **학습용 데이터(Train):** 모델이 패턴과 규칙을 배울 수 있도록 제공하는 데이터입니다.
* **테스트용 데이터(Test):** 모델이 한 번도 보지 못한 상황에서 얼마나 잘 맞히는지 검증하는 시험지 역할을 합니다.

In [9]:
# Train/Test 분리 필요성 안내 메시지
print("💡 Train / Test 데이터 분리 원칙:")
print("1. 모델 학습에 사용된 데이터로 평가하면 과적합(Overfitting)을 감지할 수 없습니다.")
print("2. 평가용 데이터(Test)는 모델 학습 과정에 절대로 노출되어서는 안 됩니다.")

💡 Train / Test 데이터 분리 원칙:
1. 모델 학습에 사용된 데이터로 평가하면 과적합(Overfitting)을 감지할 수 없습니다.
2. 평가용 데이터(Test)는 모델 학습 과정에 절대로 노출되어서는 안 됩니다.


### 💡 실습 결과 상세 정리
* 머신러닝 평가의 객관성을 확보하기 위해 Train 데이터와 Test 데이터를 엄격히 분리하여 실험을 진행하기로 결정했습니다.

## 실습 10. train / test 데이터 나누기

### 📌 학습 목표 및 개념
* **개념:** `train_test_split` 함수를 사용하여 전체 데이터셋($X$, $y$)을 모델 학습용(Train)과 최종 평가용(Test)으로 분리합니다.
* **학습 목적:** 데이터셋의 20%를 테스트용으로 지정하고, `stratify` 옵션을 통해 클래스 비율을 균등하게 유지하며 분할하는 방법을 익힙니다.

In [11]:
# stratify=y 옵션만 제거하고 실행
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train 크기:", X_train.shape)
print("X_test 크기 :", X_test.shape)
print("y_train 크기:", y_train.shape)
print("y_test 크기 :", y_test.shape)

X_train 크기: (159,)
X_test 크기 : (40,)
y_train 크기: (159,)
y_test 크기 : (40,)


### 💡 실습 결과 상세 정리
* `test_size=0.2`: 전체 데이터의 20%를 최종 평가용(Test)으로 분리했습니다.
* `random_state=42`: 실험 결과를 항상 동일하게 재현할 수 있도록 고정했습니다.
* `stratify=y`: 학습용 데이터와 테스트용 데이터의 분야별 비율을 비슷하게 유지하도록 설정했습니다.

## 실습 11. stratify가 필요한 이유 확인하기

### 📌 학습 목표 및 개념
* **개념:** `train_test_split` 시 `stratify=y` 옵션을 사용하여 원본 데이터의 분야별 비율을 학습용(Train)과 테스트용(Test) 데이터셋에 균등하게 유지합니다[cite: 1].
* **학습 목적:** 특정 분야가 한쪽 데이터셋에 쏠리는 현상을 방지하여 모델 평가의 신뢰성을 확보합니다[cite: 1].

In [12]:
# 전체 vs Train vs Test 분야별 비율 비교
print("전체")
print(y.value_counts(normalize=True).head())

print("\nTrain")
print(y_train.value_counts(normalize=True).head())

print("\nTest")
print(y_test.value_counts(normalize=True).head())

전체
분야
소설       0.241206
인문       0.135678
경제/경영    0.130653
시/에세이    0.125628
자기계발     0.085427
Name: proportion, dtype: float64

Train
분야
소설       0.251572
인문       0.144654
시/에세이    0.138365
경제/경영    0.106918
외국어      0.088050
Name: proportion, dtype: float64

Test
분야
경제/경영      0.225
소설         0.200
인문         0.100
자기계발       0.100
어린이(초등)    0.075
Name: proportion, dtype: float64


### 💡 실습 결과 상세 정리
* `stratify=y` 옵션을 통해 원본 데이터, Train 데이터, Test 데이터 간의 카테고리별 데이터 분포 비율이 비슷하게 유지됨을 확인했습니다[cite: 1].

## 실습 12. 가장 중요한 원칙: split을 먼저 한다

### 📌 학습 목표 및 개념
* **개념:** 머신러닝 평가 시 가장 흔하게 발생하는 **데이터 누수(Data Leakage)**를 방지하기 위해 반드시 데이터를 Train/Test로 먼저 분할한 후 TF-IDF 변환을 수행해야 함을 이해합니다.
* **학습 목적:** 평가 데이터(Test Set)의 단어 정보나 가중치가 모델 학습 과정에 유입되지 않는 올바른 전처리 순서를 정립합니다.

---

### ❌ 잘못된 순서 (Data Leakage 발생)
1. 전체 데이터에 TF-IDF `fit_transform()` 적용
2. Train / Test 데이터 분할
3. 모델 학습 및 평가
* **문제점:** Test 데이터의 단어 정보와 IDF 가중치가 이미 Vectorizer 학습에 반영되어, 실제 상황보다 평가 결과가 과도하게 높게 나오는 착시 현상이 발생합니다.

### ✅ 올바른 순서
1. 원본 텍스트 데이터 Train / Test 분할 (`train_test_split`)
2. Train 데이터에만 TF-IDF `fit_transform()` 적용 (단어 사전 및 IDF 학습)
3. Test 데이터는 학습된 기준에 맞춰 `transform()`만 적용
4. 모델 학습 및 평가

### 💡 실습 결과 상세 정리
* 데이터 누수를 방지하기 위해 TF-IDF Vectorizer 학습 시점은 항상 Train/Test 분할 이후여야 한다는 핵심 원칙을 확인했습니다.

## 실습 13. fit과 transform 다시 확인하기

### 📌 학습 목표 및 개념
* **개념:** `fit()`과 `transform()`의 역할 분담을 명확히 이해합니다.
* **학습 목적:** Vectorizer가 학습 데이터에서 어떤 규칙을 배우고(`fit`), 그 규칙을 데이터에 어떻게 적용치(`transform`)를 구분하여 올바른 전처리 코드를 작성합니다.

---

### 💡 fit() vs transform() 역할 구분
* **`fit()` (규칙 학습):**
  * Train 데이터만 대상으로 수행합니다.
  * 어휘 사전(Vocabulary) 생성, 문서 빈도(DF) 및 IDF 가중치를 계산합니다.
* **`transform()` (규칙 적용):**
  * 학습된 어휘 사전과 IDF 기준을 바탕으로 텍스트 데이터를 수치 행렬로 변환합니다.
  * Test 데이터에는 절대 `fit()`을 다시 호출하지 않고 `transform()`만 수행하여 새로운 단어가 유입되더라도 기존 모델 기준을 유지합니다.

### 💡 실습 결과 상세 정리
* `fit()`과 `transform()`의 기능적 차이를 이해하였으며, Test 데이터의 일반화 평가 기준을 유지하기 위한 적용 방식을 확립했습니다.

## 실습 14. TF-IDF Vectorizer 만들기

### 📌 학습 목표 및 개념
* **개념:** scikit-learn의 `TfidfVectorizer` 객체를 생성하여 텍스트를 수치형 피처 벡터로 변환할 준비를 합니다.
* **학습 목적:** 복잡한 파라미터 설정을 넣기 전에 기본(Baseline) 객체를 먼저 생성하여 텍스트 전처리 프로세스의 기준점을 마련합니다.

In [13]:
# TF-IDF Vectorizer 기본 객체 생성
tfidf = TfidfVectorizer()

print("TF-IDF Vectorizer 객체 생성 완료:", tfidf)

TF-IDF Vectorizer 객체 생성 완료: TfidfVectorizer()


In [14]:
# TF-IDF Vectorizer 기본 객체 생성
tfidf = TfidfVectorizer()

# 옵션 및 주요 설정값을 표(DataFrame)로 출력
df_tfidf_info = pd.DataFrame(
    list(tfidf.get_params().items()), 
    columns=['파라미터(Parameter)', '설정값(Value)']
)

# 표 형태로 바로 출력 (print 쓰지 않음)
df_tfidf_info.head(10)

,파라미터(Parameter),설정값(Value)
0,analyzer,word
1,binary,False
2,decode_error,strict
3,dtype,<class 'numpy.float64'>
4,encoding,utf-8
5,input,content
6,lowercase,True
7,max_df,1.0
8,max_features,None
9,min_df,1


### 💡 실습 결과 상세 정리
* `min_df`, `max_df` 등의 세부 옵션을 적용하기 전, 기본 설정의 `TfidfVectorizer` 객체를 성공적으로 정의했습니다.

## 실습 15. train 데이터에만 fit_transform 적용하기

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)

이 한 줄에서 두 작업이 수행됩니다[cite: 1].
> 1. X_train에서 단어 사전과 IDF 학습
> 2. X_train을 TF-IDF 행렬로 변환[cite: 1]

행렬 크기를 확인합니다[cite: 1].

In [ ]:
print("Train TF-IDF shape:", X_train_tfidf.shape)

행은 train 문서 수입니다[cite: 1].
열은 train 데이터에서 만들어진 vocabulary의 단어 수입니다[cite: 1].